In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


# MLFlow Initialization

In [2]:
%pip install -q dagshub mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 630.0 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 3.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 41.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 82.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

In [3]:
import dagshub
dagshub.init(repo_owner='izere23', repo_name='Assignment-2-IEEE-CIS-Fraud-Detection', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=5bf4a409-b56a-442c-9a78-14b377c2a861&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=87b730bcdca21f52f53900f1e539c5fc49095db797d94ab8047173d2948359c8




Output()

Accessing as izere23

Initialized MLflow to track repo "izere23/Assignment-2-IEEE-CIS-Fraud-Detection"

Repository izere23/Assignment-2-IEEE-CIS-Fraud-Detection initialized!

In [4]:
import mlflow

# Data Loading

In [5]:
BASE_PATH = '/kaggle/input/competitions/ieee-fraud-detection/'
train_transaction = pd.read_csv(BASE_PATH + "train_transaction.csv")
train_identity = pd.read_csv(BASE_PATH + "train_identity.csv")

train = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

print(train.shape)
print(train["isFraud"].value_counts(normalize=True))

(590540, 434)
isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64


In [6]:
from sklearn.model_selection import train_test_split

X = train.drop(columns=["isFraud"])
y = train["isFraud"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_val.shape)
print(y_train.mean(), y_val.mean())

(472432, 433) (118108, 433)
0.03498916246147594 0.0349933958749619


# Cleaning

## High Missing Columns

In [7]:
from sklearn.base import BaseEstimator, TransformerMixin

class HighMissingDropper(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.98):
        self.threshold = threshold

    def fit(self, X, y=None):
        self.cols_to_drop_ = X.columns[X.isna().mean() > self.threshold].tolist()
        return self

    def transform(self, X):
        X = X.copy()
        return X.drop(columns=self.cols_to_drop_, errors="ignore")

## High Dominance Columns

In [19]:
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np

class HighDominanceDropper(BaseEstimator, TransformerMixin):
    def __init__(self, dominance_threshold):
        self.dominance_threshold = dominance_threshold

    def fit(self, X, y=None):
        X = X.copy()

        dominance = X.apply(
            lambda col: col.value_counts(normalize=True, dropna=False).iloc[0]
            if col.value_counts(dropna=False).shape[0] > 0 else 1
        )

        self.cols_to_drop_ = dominance[
            dominance > self.dominance_threshold
        ].index.tolist()

        return self

    def transform(self, X):
        X = X.copy()
        return X.drop(columns=self.cols_to_drop_, errors="ignore")

In [20]:
cleaning_pipeline = Pipeline([
    ("high_dominacne_columns", HighDominanceDropper(dominance_threshold=0.995))
])

# Feature Engineering

## Log Adder

In [10]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np


class LogAmountAdder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        if "TransactionAmt" in X.columns:
            X["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])

        return X

## UID

In [23]:
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np


class UIDAmountDeviationAdder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):

        X_temp = X.copy()

        X_temp["uid"] = (
            X_temp["card1"].astype(str) + "_" +
            X_temp["addr1"].astype(str)
        )

        self.uid_mean_ = (
            X_temp.groupby("uid")["TransactionAmt"]
            .mean()
            .to_dict()
        )

        return self

    def transform(self, X):

        X = X.copy()

        X["uid"] = (
            X["card1"].astype(str) + "_" +
            X["addr1"].astype(str)
        )

        X["uid_mean_amt"] = (
            X["uid"]
            .map(self.uid_mean_)
        )

        X["uid_amount_deviation"] = (
            X["TransactionAmt"] -
            X["uid_mean_amt"]
        )

        X.drop(columns=["uid"], inplace=True)

        return X

In [24]:
feature_engineering_pipeline = Pipeline([
    ("uid_deviation", UIDAmountDeviationAdder())
])

# Preprocessing

In [11]:
class AutoPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.preprocessor_ = None

    def fit(self, X, y=None):
        num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
        cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

        num_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ])

        cat_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ])

        self.preprocessor_ = ColumnTransformer([
            ("num", num_pipeline, num_cols),
            ("cat", cat_pipeline, cat_cols)
        ])

        self.preprocessor_.fit(X, y)
        return self

    def transform(self, X):
        return self.preprocessor_.transform(X)

# Random Forest Pipeline

In [25]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=700,
    max_depth=18,
    min_samples_split=18,
    min_samples_leaf=3,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ("feature_engineering", feature_engineering_pipeline),
    ("cleaning", cleaning_pipeline),
    ("preprocessing", AutoPreprocessor()),
    ("model", model)
])

# Training

In [26]:
import mlflow
import mlflow.sklearn
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, log_loss

mlflow.set_experiment("Random_Forest_Training")

with mlflow.start_run(run_name="RF_UID"):

    threshold = 0.5

    pipeline.fit(X_train, y_train)

    train_pred_proba = pipeline.predict_proba(X_train)[:, 1]
    val_pred_proba = pipeline.predict_proba(X_val)[:, 1]

    val_pred = (val_pred_proba >= threshold).astype(int)

    train_roc_auc = roc_auc_score(y_train, train_pred_proba)
    val_roc_auc = roc_auc_score(y_val, val_pred_proba)

    train_log_loss = log_loss(y_train, train_pred_proba)
    val_log_loss = log_loss(y_val, val_pred_proba)

    overfit_gap = train_roc_auc - val_roc_auc
    overfit_loss_gap = val_log_loss - train_log_loss

    precision = precision_score(y_val, val_pred)
    recall = recall_score(y_val, val_pred)
    f1 = f1_score(y_val, val_pred)

    mlflow.log_param("model", "RandomForestClassifier")
    mlflow.log_param("baseline_type", "raw_data_minimal_preprocessing")
    mlflow.log_param("cleaning", "False")

    mlflow.log_param("numeric_imputation", "median")
    mlflow.log_param("categorical_imputation", "most_frequent")
    mlflow.log_param("encoding", "OHE")

    mlflow.log_param("n_estimators", 700)
    mlflow.log_param("max_depth", 18)
    mlflow.log_param("min_samples_split", 18)
    mlflow.log_param("min_samples_leaf", 3)
    mlflow.log_param("max_features", "sqrt")

    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("random_state", 42)
    mlflow.log_param("n_jobs", -1)

    mlflow.log_param("feature_engineering", "True")
    mlflow.log_param("feature_selection", "False")
    mlflow.log_param("threshold", threshold)

    mlflow.log_metric("train_roc_auc", train_roc_auc)
    mlflow.log_metric("val_roc_auc", val_roc_auc)
    mlflow.log_metric("overfit_gap", overfit_gap)

    mlflow.log_metric("train_log_loss", train_log_loss)
    mlflow.log_metric("val_log_loss", val_log_loss)
    mlflow.log_metric("overfit_loss_gap", overfit_loss_gap)

    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    mlflow.sklearn.log_model(
        pipeline,
        artifact_path="random_forest"
    )

    print("Train ROC-AUC:", train_roc_auc)
    print("Val ROC-AUC:", val_roc_auc)
    print("Overfit Gap:", overfit_gap)
    print("Train Log Loss:", train_log_loss)
    print("Val Log Loss:", val_log_loss)
    print("Overfit Loss Gap:", overfit_loss_gap)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1:", f1)

2026/05/07 00:41:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 00:41:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Train ROC-AUC: 0.9478357866219023
Val ROC-AUC: 0.9121355211216522
Overfit Gap: 0.03570026550025007
Train Log Loss: 0.2926397115415166
Val Log Loss: 0.3002180840480541
Overfit Loss Gap: 0.007578372506537534
Precision: 0.33275159976730656
Recall: 0.6919912896201307
F1: 0.44940289126335636
🏃 View run RF_UID at: https://dagshub.com/izere23/Assignment-2-IEEE-CIS-Fraud-Detection.mlflow/#/experiments/3/runs/3fed0370cf4749319c19c31faaf5c397
🧪 View experiment at: https://dagshub.com/izere23/Assignment-2-IEEE-CIS-Fraud-Detection.mlflow/#/experiments/3
